In [1]:
"""
Burstiness Features Extraction (Coordination Detection)
========================================================
检测时间同步性/突发性，识别协同操纵行为

输入: reddit_wsb_for_network.csv
输出: burstiness_features_5min.parquet

核心特征（6列）:
1. burstiness_coefficient - 突发系数 (接近1=高度突发)
2. sync_posting_rate - 同步发帖率 (1min peak / 5min avg)
3. inter_arrival_mean - 平均发帖间隔
4. inter_arrival_std - 发帖间隔标准差
5. inter_arrival_min - 最小发帖间隔
6. pct_posts_under_5sec - <5秒发帖比例（异常快速）

理论依据:
- Kumar et al. (2018) "Detecting Suspicious Accounts on Social Media"
- Bot/协同者: 高burstiness, 低inter-arrival time
- 有机讨论: 低burstiness, 随机分布
"""

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================================
# 参数设置
# ============================================================

TIME_WINDOW = 5  # 分钟
DATA_FILE = 'reddit_wsb_for_network.csv'

# ============================================================
# 核心函数
# ============================================================

def calculate_burstiness_coefficient(inter_arrivals):
    """
    计算Burstiness Coefficient
    B = (σ - μ) / (σ + μ)
    
    B ≈ 1: 高度突发 (bursty)
    B ≈ 0: 均匀分布 (regular)
    B ≈ -1: 周期性 (periodic)
    """
    if len(inter_arrivals) < 2:
        return 0
    
    mean = np.mean(inter_arrivals)
    std = np.std(inter_arrivals)
    
    if mean + std == 0:
        return 0
    
    B = (std - mean) / (std + mean)
    
    return B


def extract_burstiness_features(window_data):
    """
    从时间窗口数据中提取burstiness特征
    """
    
    if len(window_data) == 0:
        return {
            'burstiness_coefficient': 0,
            'sync_posting_rate': 0,
            'inter_arrival_mean': 0,
            'inter_arrival_std': 0,
            'inter_arrival_min': 0,
            'pct_posts_under_5sec': 0
        }
    
    # 排序
    sorted_times = window_data['timestamp'].sort_values()
    
    # ========== 1. Inter-arrival Times ==========
    if len(sorted_times) > 1:
        inter_arrivals = sorted_times.diff().dt.total_seconds()
        inter_arrivals = inter_arrivals[inter_arrivals.notna()].values
        
        if len(inter_arrivals) > 0:
            ia_mean = np.mean(inter_arrivals)
            ia_std = np.std(inter_arrivals)
            ia_min = np.min(inter_arrivals)
            
            # Burstiness coefficient
            B = calculate_burstiness_coefficient(inter_arrivals)
            
            # 异常快速发帖比例
            pct_under_5sec = (inter_arrivals < 5).sum() / len(inter_arrivals) * 100
        else:
            ia_mean = 0
            ia_std = 0
            ia_min = 0
            B = 0
            pct_under_5sec = 0
    else:
        ia_mean = 0
        ia_std = 0
        ia_min = 0
        B = 0
        pct_under_5sec = 0
    
    # ========== 2. Sync Posting Rate ==========
    # 计算1分钟窗口的最大发帖数 vs 5分钟平均
    window_data['minute_bin'] = window_data['timestamp'].dt.floor('1min')
    posts_per_minute = window_data.groupby('minute_bin').size()
    
    if len(posts_per_minute) > 0:
        max_1min = posts_per_minute.max()
        avg_per_min = len(window_data) / 5  # 5分钟窗口
        
        if avg_per_min > 0:
            sync_rate = max_1min / avg_per_min
        else:
            sync_rate = 0
    else:
        sync_rate = 0
    
    return {
        'burstiness_coefficient': B,
        'sync_posting_rate': sync_rate,
        'inter_arrival_mean': ia_mean,
        'inter_arrival_std': ia_std,
        'inter_arrival_min': ia_min,
        'pct_posts_under_5sec': pct_under_5sec
    }


# ============================================================
# 主函数
# ============================================================

def main():
    start_time = datetime.now()
    
    print("\n" + "="*70)
    print("BURSTINESS FEATURES EXTRACTION (COORDINATION DETECTION)")
    print("="*70)
    
    # ========== Step 1: 加载数据 ==========
    print("\nStep 1: Loading Reddit data...")
    
    try:
        reddit_df = pd.read_csv(DATA_FILE)
        reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'])
        
        print(f"  ✓ Loaded {len(reddit_df):,} items")
        print(f"  Date range: {reddit_df['timestamp'].min()} to {reddit_df['timestamp'].max()}")
        
    except FileNotFoundError:
        print(f"  ✗ {DATA_FILE} not found!")
        return
    
    # ========== Step 2: 创建时间窗口 ==========
    print("\nStep 2: Creating time windows...")
    
    reddit_df['time_window'] = reddit_df['timestamp'].dt.floor(f'{TIME_WINDOW}min')
    time_windows = sorted(reddit_df['time_window'].unique())
    
    print(f"  ✓ Created {len(time_windows):,} time windows")
    
    # ========== Step 3: 逐窗口提取特征 ==========
    print("\nStep 3: Extracting burstiness features...")
    
    features_list = []
    
    for i, window_time in enumerate(time_windows):
        if i % 100 == 0:
            print(f"  Processing window {i+1:,}/{len(time_windows):,}...", end='\r')
        
        # 获取该窗口的数据
        window_data = reddit_df[reddit_df['time_window'] == window_time].copy()
        
        # 提取特征
        window_features = extract_burstiness_features(window_data)
        window_features['timestamp'] = window_time
        
        features_list.append(window_features)
    
    print(f"\n  ✓ Extracted features for {len(features_list):,} windows")
    
    # 转换为DataFrame
    burstiness_df = pd.DataFrame(features_list)
    
    # ========== Step 4: 统计分析 ==========
    print("\nStep 4: Feature statistics...")
    
    print(f"\n  Burstiness coefficient:")
    print(f"    Mean: {burstiness_df['burstiness_coefficient'].mean():.3f}")
    print(f"    Std: {burstiness_df['burstiness_coefficient'].std():.3f}")
    print(f"    Max: {burstiness_df['burstiness_coefficient'].max():.3f}")
    print(f"    >0.5 (high burst): {(burstiness_df['burstiness_coefficient'] > 0.5).sum():,} windows")
    
    print(f"\n  Sync posting rate:")
    print(f"    Mean: {burstiness_df['sync_posting_rate'].mean():.2f}x")
    print(f"    Max: {burstiness_df['sync_posting_rate'].max():.1f}x")
    print(f"    >3x: {(burstiness_df['sync_posting_rate'] > 3).sum():,} windows (suspicious)")
    print(f"    >5x: {(burstiness_df['sync_posting_rate'] > 5).sum():,} windows (highly suspicious)")
    
    print(f"\n  Inter-arrival time (seconds):")
    # 只看有活动的窗口
    active = burstiness_df[burstiness_df['inter_arrival_mean'] > 0]
    if len(active) > 0:
        print(f"    Mean: {active['inter_arrival_mean'].mean():.1f}")
        print(f"    Min observed: {active['inter_arrival_min'].min():.1f}")
    
    print(f"\n  Fast posting (<5 sec):")
    print(f"    Mean %: {burstiness_df['pct_posts_under_5sec'].mean():.1f}%")
    print(f"    Max %: {burstiness_df['pct_posts_under_5sec'].max():.1f}%")
    suspicious_fast = (burstiness_df['pct_posts_under_5sec'] > 20).sum()
    print(f"    Windows with >20%: {suspicious_fast:,} (potential bot/coordination)")
    
    # ========== Step 5: 保存 ==========
    print("\nStep 5: Saving...")
    
    burstiness_df.to_parquet('burstiness_features_5min.parquet', index=False)
    
    print(f"  ✓ Saved: burstiness_features_5min.parquet")
    print(f"  Shape: {burstiness_df.shape}")
    
    # ========== 总结 ==========
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    print("\n" + "="*70)
    print("COMPLETE")
    print("="*70)
    
    print(f"\nTime windows: {len(burstiness_df):,}")
    print(f"Feature columns: {len(burstiness_df.columns)}")
    
    print(f"\nFeatures extracted:")
    print(f"  ✓ burstiness_coefficient (coordination indicator)")
    print(f"  ✓ sync_posting_rate (burst intensity)")
    print(f"  ✓ inter_arrival_mean, std, min (timing patterns)")
    print(f"  ✓ pct_posts_under_5sec (bot detection)")
    
    print(f"\nCoordination signals detected:")
    high_burst = (burstiness_df['burstiness_coefficient'] > 0.5).sum()
    high_sync = (burstiness_df['sync_posting_rate'] > 3).sum()
    fast_posts = (burstiness_df['pct_posts_under_5sec'] > 20).sum()
    
    print(f"  Windows with high burstiness (>0.5): {high_burst:,}")
    print(f"  Windows with high sync rate (>3x): {high_sync:,}")
    print(f"  Windows with fast posting (>20%): {fast_posts:,}")
    
    total_suspicious = len(burstiness_df[
        (burstiness_df['burstiness_coefficient'] > 0.5) |
        (burstiness_df['sync_posting_rate'] > 3) |
        (burstiness_df['pct_posts_under_5sec'] > 20)
    ])
    
    print(f"\n  Total suspicious windows: {total_suspicious:,} ({total_suspicious/len(burstiness_df)*100:.1f}%)")
    
    print(f"\nRuntime: {duration:.1f} seconds")
    
    print("\n✓ Burstiness feature extraction complete!")
    print("  (Alignment will be done in merge_all_features.py)")
    
    # ========== 解释 ==========
    print("\n" + "="*70)
    print("INTERPRETATION GUIDE")
    print("="*70)
    
    print("\nCoordination indicators:")
    print("  • Burstiness > 0.5: Strong burst pattern (organized posting)")
    print("  • Sync rate > 3x: Concentrated activity (possible coordination)")
    print("  • Fast posting > 20%: Bot-like behavior")
    
    print("\nNormal discussion patterns:")
    print("  • Burstiness ≈ 0: Random, organic timing")
    print("  • Sync rate < 2x: Natural fluctuation")
    print("  • Fast posting < 10%: Human-paced interaction")


if __name__ == "__main__":
    main()


BURSTINESS FEATURES EXTRACTION (COORDINATION DETECTION)

Step 1: Loading Reddit data...
  ✓ Loaded 1,606,093 items
  Date range: 2019-07-01 04:35:02 to 2021-06-29 23:58:28

Step 2: Creating time windows...
  ✓ Created 77,267 time windows

Step 3: Extracting burstiness features...
  Processing window 77,201/77,267...
  ✓ Extracted features for 77,267 windows

Step 4: Feature statistics...

  Burstiness coefficient:
    Mean: -0.084
    Std: 0.175
    Max: 0.348
    >0.5 (high burst): 0 windows

  Sync posting rate:
    Mean: 2.81x
    Max: 5.0x
    >3x: 25,508 windows (suspicious)
    >5x: 0 windows (highly suspicious)

  Inter-arrival time (seconds):
    Mean: 43.7
    Min observed: 0.0

  Fast posting (<5 sec):
    Mean %: 15.7%
    Max %: 100.0%
    Windows with >20%: 22,279 (potential bot/coordination)

Step 5: Saving...
  ✓ Saved: burstiness_features_5min.parquet
  Shape: (77267, 7)

COMPLETE

Time windows: 77,267
Feature columns: 7

Features extracted:
  ✓ burstiness_coefficient 